In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
import numpy as np

In [25]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize model
model = SentenceTransformer("all-MiniLM-L6-v2")

text = """
The Amazon rainforest is one of the largest tropical forests in the world.
It is home to millions of species of plants, animals, and insects.
The forest plays a crucial role in regulating the Earth's climate.

Basketball is a popular sport played by two teams of five players.
The objective of the game is to score points by shooting the ball through the opponent's hoop.
Professional basketball leagues like the NBA attract millions of fans worldwide.

Artificial intelligence is transforming many industries including healthcare and finance.
Machine learning algorithms can analyze large amounts of data to find patterns.
AI systems are used in applications such as recommendation systems and self-driving cars.
"""

# Split sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# Create embeddings
embeddings = model.encode(sentences)

threshold = 0.2   # lowered threshold

chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):

    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]

    print(f"Similarity {i-1}-{i}: {sim:.3f}")   # debug similarity

    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

chunks.append(" ".join(current_chunk))

print("\n📌 Semantic Chunks:\n")

for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}:")
    print(chunk)
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8057.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity 0-1: 0.400
Similarity 1-2: 0.368
Similarity 2-3: -0.036
Similarity 3-4: 0.461
Similarity 4-5: 0.336
Similarity 5-6: 0.167
Similarity 6-7: 0.407
Similarity 7-8: 0.369

📌 Semantic Chunks:

Chunk 1:
The Amazon rainforest is one of the largest tropical forests in the world. It is home to millions of species of plants, animals, and insects. The forest plays a crucial role in regulating the Earth's climate.

Chunk 2:
Basketball is a popular sport played by two teams of five players. The objective of the game is to score points by shooting the ball through the opponent's hoop. Professional basketball leagues like the NBA attract millions of fans worldwide.

Chunk 3:
Artificial intelligence is transforming many industries including healthcare and finance. Machine learning algorithms can analyze large amounts of data to find patterns. AI systems are used in applications such as recommendation systems and self-driving cars.



In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# Step 1: Load document
loader = TextLoader("langchain_intro.txt")
docs = loader.load()

# Step 2: Initialize embedding model
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 3: Semantic chunking (topic boundaries)
semantic_chunker = SemanticChunker(
    embedding,
    breakpoint_threshold_type="standard_deviation"
)

semantic_chunks = semantic_chunker.split_documents(docs)

# Step 4: Sliding window chunking (size control)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

hybrid_chunks = text_splitter.split_documents(semantic_chunks)

# Step 5: Print results
for i, chunk in enumerate(hybrid_chunks):
    print(f"\nChunk {i+1}:\n")
    print(chunk.page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6567.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk 1:

Cats are independent animals. Dogs are loyal companions. Both cats and dogs are popular pets. Asia is the largest continent. Africa has diverse wildlife. Both continents have rich cultures.
